In [8]:
!pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [30]:
import zipfile
from pathlib import Path
from io import BytesIO  # Import BytesIO
from PyPDF2 import PdfReader  # No PdfReadError in the latest PyPDF2


In [31]:

def process_file(file_path):
    # Get the extension of the submitted file
    file_extension = Path(file_path).suffix

    if file_extension == '.zip':
        # Open the zip file and process its contents
        file_contents = {}
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            for file_name in zip_ref.namelist():
                if file_name.endswith('.pdf') or file_name.endswith('.py'):
                    with zip_ref.open(file_name) as file:
                        # Read as binary content for PdfReader compatibility
                        content = store_file(file.read(), file_name)
                        file_contents[file_name] = content
        return file_contents

    elif file_extension in ['.pdf', '.py']:
        # If it's a .pdf or .py file, store its content directly
        with open(file_path, 'rb') as file:
            content = store_file(file.read(), file_path)
        return {file_path: content}
    else:
        print("Unsupported file type.")
        return None


def store_file(file_content, file_name):
    # Check file extension to determine the handling method
    if file_name.endswith('.pdf'):
        return extract_pdf_text(file_content)
    elif file_name.endswith('.py'):
        return read_python_file(file_content)


def extract_pdf_text(file_content):
    # Read the content from the PDF file-like object
    try:
        reader = PdfReader(BytesIO(file_content))  # Use BytesIO for file-like object
        text = ''
        for page in reader.pages:
            text += page.extract_text() or ""  # Safely handle None
        print("Extracted text from PDF.")
        return text
    except Exception as e:  # Handle general exceptions
        print(f"Error reading PDF: {e}")
        return f"Error: Unable to read PDF."


def read_python_file(file_content):
    # Read the Python (.py) file content
    encodings = ['utf-8', 'ISO-8859-1', 'latin-1']
    for encoding in encodings:
        try:
            content = file_content.decode(encoding)
            print(f"Successfully read Python file with {encoding} encoding.")
            return content
        except UnicodeDecodeError:
            print(f"Failed to read Python file with {encoding} encoding. Trying next...")
    raise UnicodeDecodeError("Unable to read Python file with any encoding.")




In [41]:

# Example usage
file_path = 'test.zip'  # Replace with the actual file path
file_contents = process_file(file_path)
print(file_contents)


Extracted text from PDF.
Error reading PDF: EOF marker not found
{'test/Test.pdf': 'FYP Logbook Entry Sheet   FYP Logbook Entry Sheet  Meeting No: 5      Date: 10 November 2024  Start Time: 11:00 AM     End Time: 1:00 PM   Items Discussed:  a. Successful setup and connection of Milvus on the local device. b. Selection of Milvus as the vector database for the project. Implementation of sentence embeddings using Sentence-BERT.  c. Progress in extracting text from PDFs and code files.  d. The current frontend is implemented using Streamlit, but it will be changed to HTML    and CSS later for better customization.   Achievements:  a. Successfully created the frontend using Streamlit.  b. Finalized and connected Milvus as the vector database.  c. Generated embeddings using Sentence-BERT for input text.  d. Extracted text from PDF and code files.   Problems:  a. Faced challenges while connecting to the Milvus database, but they were resolved.  b. Extraction from zip files is still pending.  